# 1 - Importação das Bibliotecas

O notebook irá realizar uso da base de dados que está localizado no google drive.

Também será utilizadas bibliotecas do numpy, pandas e matplotlib para manipulação dos dados e visualização.

O notebook tdqm será utilizado para construção da barra de progresso.

O tensorflow será utilizado para construção e utilização do modelo MLP.

Já o sklearn será utilizado para os modelos MLP, RandomForest.

Por fim, o xgboost será utilizado para implementação do modelo XGBoost.

In [1]:
from google.colab import drive

import os
import gc
from datetime import datetime
import time

import numpy as np
import pandas as pd
from math import trunc


import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

import shap

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2

import joblib



In [2]:
# Checagem de disponibilidade da GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ {len(gpus)} GPU(s) disponível(is): {[gpu.name for gpu in gpus]}")
else:
    print("⚠️ Nenhuma GPU disponível. Usando CPU.")

⚠️ Nenhuma GPU disponível. Usando CPU.


In [ ]:
DIRETORIO_BASE = "."

# Função de Criação de Valores SHAP

In [ ]:
def cria_valores_shap(df, modelo_nome, nome_grupo, n_splits=5):

    X = df.drop(columns=['classe']).values.astype(np.int8)
    y = df['classe'].astype(np.int8).values

    print(f"\nIniciando avaliação para o grupo de features: '{nome_grupo}' "
          f"com {X.shape[1]} atributos e {X.shape[0]} instâncias.")

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    folds = list(skf.split(X, y))

    CAMINHO_SAIDA_BASE = (
        f'{DIRETORIO_BASE}/src_shap/{nome_grupo}/'
        f'{modelo_nome}/'
    )
    os.makedirs(CAMINHO_SAIDA_BASE, exist_ok=True)

    caminho_log = os.path.join(
        CAMINHO_SAIDA_BASE,
        f"log_tempo_shap_{nome_grupo}_{modelo_nome}.csv"
    )

    registros_log = []

    for fold in range(n_splits):

        tempo_fold_inicio = time.perf_counter()

        train_idx, test_idx = folds[fold]

        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        feature_names = np.array([c for c in df.columns if c != 'classe'])
        index_explicado = df.iloc[test_idx].index.to_numpy().astype(np.int64)


        print(f"\nFold {fold+1}/{n_splits}")
        print(f"Treino: {X_train.shape} | Teste: {X_test.shape}")
        print(f"Carregando o modelo {modelo_nome}.")

        CAMINHO_PASTA_MODELO = (
            f'{DIRETORIO_BASE}/src_inicial/{nome_grupo}/'
            f'{modelo_nome}/resultados/'
        )
        CAMINHO_MODELO = os.path.join(
            CAMINHO_PASTA_MODELO,
            f'{nome_grupo}__{modelo_nome}__fold{fold+1}.joblib'
        )

        tempo_modelo_inicio = time.perf_counter()
        modelo = joblib.load(CAMINHO_MODELO)
        tempo_modelo_fim = time.perf_counter()

        tempo_explainer_inicio = time.perf_counter()
        explainer = shap.TreeExplainer(modelo, X_train)
        tempo_explainer_fim = time.perf_counter()

        tempo_shap_inicio = time.perf_counter()
        shap_values = explainer(X_test)
        tempo_shap_fim = time.perf_counter()

        tempo_conversao_inicio = time.perf_counter()

        shap_exp = shap_values
        shap_mat = shap_exp.values.astype(np.float32)
        base_values = np.array(shap_exp.base_values).astype(np.float32)
        X_explicado = shap_exp.data.astype(np.float32)

        tempo_conversao_fim = time.perf_counter()

        saida_npz = os.path.join(
            CAMINHO_SAIDA_BASE,
            f"{nome_grupo}__{modelo_nome}__fold{fold+1}_SHAP.npz"
        )

        tempo_salvamento_inicio = time.perf_counter()

        np.savez_compressed(
            saida_npz,
            shap_values=shap_mat,
            base_values=base_values,
            feature_names=feature_names,
            X_explicado=X_explicado,
            test_index=index_explicado,
            y_test=y_test.astype(np.int32)
        )

        tempo_salvamento_fim = time.perf_counter()
        tempo_fold_fim = time.perf_counter()

        registro = {
            "data_hora": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "nome_grupo": nome_grupo,
            "modelo_nome": modelo_nome,
            "fold": fold + 1,
            "n_features": X_train.shape[1],
            "n_train": X_train.shape[0],
            "n_test": X_test.shape[0],
            "tempo_carregamento_modelo_seg": tempo_modelo_fim - tempo_modelo_inicio,
            "tempo_criacao_explainer_seg": tempo_explainer_fim - tempo_explainer_inicio,
            "tempo_calculo_shap_seg": tempo_shap_fim - tempo_shap_inicio,
            "tempo_conversao_seg": tempo_conversao_fim - tempo_conversao_inicio,
            "tempo_salvamento_seg": tempo_salvamento_fim - tempo_salvamento_inicio,
            "tempo_total_fold_seg": tempo_fold_fim - tempo_fold_inicio,
            "caminho_modelo": CAMINHO_MODELO,
            "caminho_saida_npz": saida_npz
        }

        registros_log.append(registro)

        pd.DataFrame(registros_log).to_csv(caminho_log, index=False)

        print(f"Tempo SHAP fold {fold+1}: {registro['tempo_calculo_shap_seg']:.2f} segundos")
        print(f"Tempo total fold {fold+1}: {registro['tempo_total_fold_seg']:.2f} segundos")
        print(f"Log atualizado em: {caminho_log}")

        del shap_values, shap_exp, shap_mat, base_values, X_explicado
        del explainer, modelo
        gc.collect()

# 2 - Abertura do Arquivo, recuperação dos dados e embaralhamento


In [6]:
# Acesso ao drive pessoal
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
CAMINHO_ARQUIVO = f'{DIRETORIO_BASE}/dados/mh1m_balanceadas.npz'


dados = np.load(CAMINHO_ARQUIVO, allow_pickle=True)


X = dados['data']
y = dados['classes']
colunas = dados['column_names']

In [ ]:

rng = np.random.default_rng(42)
idx_final = rng.permutation(X.shape[0])

X = X[idx_final]
y = y[idx_final]

print(f"Dados embaralhados: X={X.shape}, y={y.shape}")

Dados embaralhados: X=(159020, 23239), y=(159020,)


# 3 - Separar as colunas das features


In [ ]:
modelo_nome = "rf"

grupos = ["intents", "permissions", "opcodes", "apicalls", "permissions_opcodes", "todas"]

for nome_grupo in grupos:


  if nome_grupo == "permissions_opcodes":
      idx_permissions = [i for i, nome in enumerate(colunas) if nome.startswith("permissions::")]
      idx_opcodes = [i for i, nome in enumerate(colunas) if nome.startswith("opcodes::")]
      idx_features = idx_permissions + idx_opcodes
  elif nome_grupo == "todas":
      idx_features = range(len(colunas))
  else:
      idx_features = [i for i, nome in enumerate(colunas) if nome.startswith(f"{nome_grupo}::")]


  df    = pd.DataFrame(X[:, idx_features], columns=np.array(colunas)[idx_features])
  df['classe'] = y

  print("DataFrames criados:")

  print(f" - df: {df.shape}")

  cria_valores_shap(df, modelo_nome, nome_grupo, 5)
  del df
  gc.collect()
